In [1]:
!pip install openai jsonlines pandas tqdm

In [2]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_FOLDER = "/content/drive/MyDrive/MrWorldwide"
GRAMMAR_FILE = f"{DRIVE_FOLDER}/generated.md"
VOCAB_FILE   = f"{DRIVE_FOLDER}/vocab.json"
OUTPUT_FILE  = f"{DRIVE_FOLDER}/training_data_v3.jsonl"

Mounted at /content/drive


In [3]:
import json
import re

def parse_grammar_rules(filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        content = f.read()
    sections = re.split(r'\n(?=## )', content.strip())
    rules = []
    for section in sections:
        lines = section.strip().split('\n')
        title_line = lines[0].lstrip('#').strip()
        level_match = re.search(r'\(([AB][12])\)', title_line)
        cefr_level = level_match.group(1) if level_match else "unknown"
        title = re.sub(r'\s*\([AB][12]\)\s*', '', title_line).strip()
        rules.append({
            "title": title,
            "cefr_level": cefr_level,
            "full_text": section.strip()
        })
    return rules

def load_vocab(filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        return json.load(f)

grammar_rules = parse_grammar_rules(GRAMMAR_FILE)
vocab_entries = load_vocab(VOCAB_FILE)

print(f"Loaded {len(grammar_rules)} grammar rules")
print(f"Loaded {len(vocab_entries)} vocabulary entries")
for r in grammar_rules:
    print(f"  [{r['cefr_level']}] {r['title']}")

Loaded 26 grammar rules
Loaded 262 vocabulary entries
  [A1] Pronoms sujets et le verbe être
  [A1] Le verbe avoir et l'âge
  [A1] Articles et genre
  [A1] Présent des verbes réguliers en -er
  [A1] La négation avec ne ... pas
  [A1] Pluriel des noms
  [A1] Poser des questions : intonation, est-ce que, inversion
  [A2] Présent des verbes en -ir et en -re
  [A2] Passé composé avec avoir
  [A2] Passé composé avec être et accord
  [A2] L'imparfait : formation et principaux usages
  [A2] Futur proche (aller + infinitif)
  [A2] Les verbes pronominaux
  [A2] Accord et placement des adjectifs
  [A2] Adjectifs possessifs et démonstratifs
  [A2] Articles partitifs : du, de la, des
  [B1] Pronoms compléments directs et indirects
  [B1] Les pronoms y et en
  [B1] Pronoms relatifs : qui, que, dont, où
  [B1] Le futur simple
  [B1] Le conditionnel présent
  [B1] Le subjonctif présent : formation et emplois
  [B2] Subjonctif passé et concordance des temps
  [B2] La voix passive
  [B2] Le discours in

In [17]:
TUTOR_SYSTEM_PROMPT = """You are MrWorldwide, a Socratic French language tutor. You help students learn French by guiding them to discover correct answers themselves — never by giving direct corrections or translations.

Rules you must follow:
1. NEVER give the correct answer directly. Instead, use hints, leading questions, or mini-exercises.
2. Always reason about the student's error inside <reasoning>...</reasoning> tags. This reasoning is hidden from the student.
3. In your <reasoning> block, identify the specific error, reference the grammar rule from the provided context, and plan your teaching strategy.
4. Your visible response should be encouraging, concise, and end with a question or exercise that guides the student toward the correct form.
5. Match your language complexity to the student's CEFR level:
   - A1: respond ~80% in English, use French only for key terms being taught
   - A2: respond ~50% English, ~50% French
   - B1: respond ~80% in French, English only for grammar terminology
   - B2: respond entirely in French
6. If the student's input has no errors, praise them and provide a follow-up exercise.
7. ONLY use French and English in your responses. Never use any other language or script."""


# Language ratio instructions per CEFR level
LANGUAGE_RULES = {
    "A1": """LANGUAGE RULE (CRITICAL): The student is a BEGINNER. The visible response (after </reasoning>) MUST be approximately 80% English and 20% French.
- Explain grammar concepts in English
- Use French only for the specific words/forms being taught
- Example tone: "Great try! In French, we use the verb 'avoir' (to have) for age, not 'être' (to be). Can you try saying 'J'ai 20 ans' instead?"
- Do NOT write full sentences in French for explanations""",

    "A2": """LANGUAGE RULE (CRITICAL): The student is ELEMENTARY level. The visible response (after </reasoning>) MUST be approximately 50% English and 50% French.
- Use simple French sentences mixed with English explanations
- Example tone: "Bien essayé ! Look at the verb 'aller' — it's a movement verb. Quels verbes de mouvement utilisent 'être' au passé composé ? Can you fix it?"
- Switch between languages naturally""",

    "B1": """LANGUAGE RULE (CRITICAL): The student is INTERMEDIATE. The visible response (after </reasoning>) MUST be approximately 80% French and 20% English.
- Write mostly in French with occasional English for complex grammar terms
- Example tone: "Bonne tentative ! Regarde la règle du subjonctif — after 'il faut que', which mode do we use? Essaie de conjuguer 'venir' au subjonctif pour 'tu'."
- Use English only when a grammar concept needs clarification""",

    "B2": """LANGUAGE RULE (CRITICAL): The student is UPPER-INTERMEDIATE. The visible response (after </reasoning>) MUST be 100% in French.
- Write entirely in French, including all grammar explanations
- Example tone: "Très bien pour la structure générale ! Cependant, regarde la concordance des temps : quand le verbe introducteur est au passé, quel temps utilise-t-on dans la subordonnée ?"
- Do NOT use any English at all"""
}


META_PROMPT_GRAMMAR = """You are generating training data for a Socratic French tutor AI called MrWorldwide. Given a grammar rule below, create ONE realistic training example.

GRAMMAR RULE:
{grammar_rule}

RELEVANT VOCABULARY (use some naturally):
{vocab_sample}

STUDENT CEFR LEVEL: {cefr_level}

{language_rule}

FORMAT RULES (MANDATORY — follow exactly):
1. The assistant response MUST have exactly TWO parts separated by a closing tag:
   PART 1: <reasoning> block (3-5 sentences analyzing the error, referencing the context, planning strategy)
   PART 2: </reasoning> followed by a BLANK LINE, then the visible Socratic response (2-4 sentences)
2. The <reasoning> block must EXPLICITLY mention the grammar rule from the context
3. The visible response must NEVER contain the direct correction
4. The visible response must END with a question or exercise for the student

Generate a JSON object with exactly this structure:
{{
  "user": "[CONTEXT] <grammar rule here> [/CONTEXT]\\n\\n<student message with a realistic error related to the grammar rule>",
  "assistant": "<reasoning>\\n<3-5 sentences: identify error, reference context rule, plan teaching strategy>\\n</reasoning>\\n\\n<2-4 sentence Socratic response following the LANGUAGE RULE above>"
}}

ERROR VARIETY — use one of these error types:
- Wrong conjugation (wrong ending, wrong person)
- Wrong tense (present instead of past, etc.)
- Wrong auxiliary (avoir/être confusion)
- Gender/agreement error
- Word order mistake
- Missing accent or elision
- Direct translation from English that doesn't work in French
- Using the wrong verb entirely

EXAMPLE OUTPUT (do NOT copy this — generate a NEW one based on the grammar rule above):

{{
  "user": "[CONTEXT] Le verbe avoir sert pour la possession et pour dire l'âge. Pour exprimer l'âge on utilise la structure avoir + nombre + ans. Conjugaison: j'ai, tu as, il/elle a, nous avons, vous avez, ils/elles ont [/CONTEXT]\\n\\nJe suis 20 ans et je suis un étudiant.",
  "assistant": "<reasoning>\\nThe student wrote 'Je suis 20 ans' using être instead of avoir for age. The context confirms: age requires avoir + nombre + ans. Since this is A1, I will respond mostly in English and hint at the correct verb without giving the answer.\\n</reasoning>\\n\\nGreat try! You got 'je suis un étudiant' right. But for age, French doesn't use 'être'. Look at the verb in your context — can you spot which one is used for age? Try rewriting just the first part!"
}}

IMPORTANT:
- The student error must be REALISTIC for a {cefr_level} learner
- The <reasoning> block is written in English (it's internal, not shown to student)
- The visible response follows the LANGUAGE RULE above strictly
- Respond ONLY with the JSON object. No markdown, no backticks, no extra text."""


META_PROMPT_VOCAB = """You are generating training data for a Socratic French tutor AI called MrWorldwide. Given vocabulary items, create ONE training example where the tutor teaches or tests these words.

VOCABULARY ITEMS:
{vocab_items}

STUDENT CEFR LEVEL: {cefr_level}

{language_rule}

FORMAT RULES (MANDATORY — follow exactly):
1. The assistant response MUST have exactly TWO parts:
   PART 1: <reasoning> block (3-5 sentences)
   PART 2: </reasoning> followed by a BLANK LINE, then the visible response
2. The visible response must be a vocabulary exercise (fill-in-the-blank, matching, sentence completion, gender quiz, context clue guessing)
3. NEVER just list definitions

Generate a JSON object with exactly this structure:
{{
  "user": "[CONTEXT] <vocabulary items with definitions> [/CONTEXT]\\n\\n<student message: asking about words, using a word wrong, or requesting practice>",
  "assistant": "<reasoning>\\n<analyze what student needs, reference vocab context, plan exercise type>\\n</reasoning>\\n\\n<vocabulary exercise following the LANGUAGE RULE above>"
}}

EXAMPLE OUTPUT (do NOT copy this — generate a NEW one based on the vocabulary above):

{{
  "user": "[CONTEXT] la pomme (f) — apple — Ex: Je mange une pomme.\\nle pain (m) — bread — Ex: J'achète du pain.\\nle fromage (m) — cheese — Ex: J'aime le fromage.\\nle lait (m) — milk — Ex: Je bois du lait le matin.\\nle café (m) — coffee — Ex: Je prends un café le matin. [/CONTEXT]\\n\\nCan you give me a practice exercise with these food words?",
  "assistant": "<reasoning>\\nThe student wants to practice food vocabulary. I will create a fill-in-the-blank exercise using these items in simple sentences. Since this is A1, I will write mostly in English with the French words as the focus.\\n</reasoning>\\n\\nLet's practice! Fill in the blank with the correct word from your list:\\n\\n1. Le matin, je bois du _______ avec du sucre.\\n2. Je mange une _______ rouge après le déjeuner.\\n3. J'achète du _______ à la boulangerie.\\n\\nWhich words fit best? Look at the gender hints in your context if you need help!"
}}

IMPORTANT:
- Vary exercise types across examples
- The visible response follows the LANGUAGE RULE strictly
- Respond ONLY with the JSON object."""


In [18]:
from google.colab import userdata
OPENAI_API_KEY = userdata.get('OPEN_AI_API_KEY')

In [19]:
import openai
import jsonlines
import random
import time
from tqdm import tqdm

client = openai.OpenAI(api_key=f"{OPENAI_API_KEY}")


def validate_example(parsed, cefr_level):
    """Strict validation — returns (is_valid, reason)."""
    if "user" not in parsed or "assistant" not in parsed:
        return False, "missing user or assistant key"

    user = parsed["user"]
    assistant = parsed["assistant"]

    # Check [CONTEXT] in user
    if "[CONTEXT]" not in user or "[/CONTEXT]" not in user:
        return False, "missing [CONTEXT] block in user"

    # Check <reasoning> opens
    if "<reasoning>" not in assistant:
        return False, "missing <reasoning> tag"

    # Check </reasoning> closes
    if "</reasoning>" not in assistant:
        return False, "missing </reasoning> tag"

    # Check content after </reasoning>
    after_reasoning = assistant.split("</reasoning>")[-1].strip()
    if len(after_reasoning) < 30:
        return False, f"visible response too short ({len(after_reasoning)} chars)"

    # Check reasoning content references context
    reasoning_block = assistant.split("<reasoning>")[1].split("</reasoning>")[0]
    if len(reasoning_block.strip()) < 50:
        return False, f"reasoning too short ({len(reasoning_block.strip())} chars)"

    # Check no [/CONTEXT] leaked into assistant
    if "[/CONTEXT]" in assistant:
        return False, "[/CONTEXT] leaked into assistant response"

    # Check no Chinese characters
    if any('\u4e00' <= c <= '\u9fff' for c in assistant):
        return False, "contains Chinese characters"

    # Check language ratio (rough estimate)
    french_markers = ['je ', 'tu ', 'il ', 'elle ', 'nous ', 'vous ', 'est ',
                      'sont ', 'une ', 'des ', 'dans ', 'pour ', 'avec ',
                      'que ', 'qui ', 'sur ', 'pas ', 'mais ', 'très ']
    english_markers = ['the ', 'is ', 'are ', 'you ', 'your ', 'this ',
                       'try ', 'look ', 'think ', 'what ', 'how ', 'can ',
                       'which', 'should', 'would', 'remember', 'notice']

    after_lower = after_reasoning.lower()
    fr = sum(after_lower.count(m) for m in french_markers)
    en = sum(after_lower.count(m) for m in english_markers)
    total = fr + en
    if total > 0:
        fr_ratio = fr / total
        if cefr_level == "A1" and fr_ratio > 0.5:
            return False, f"A1 response too French ({fr_ratio:.0%})"
        if cefr_level == "B2" and fr_ratio < 0.7:
            return False, f"B2 response not French enough ({fr_ratio:.0%})"

    return True, "ok"


def call_llm(prompt, cefr_level, max_retries=3):
    """Call GPT-4o-mini with JSON mode enforced."""
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[
                    {"role": "system", "content": "You are a training data generator. Always respond with valid JSON only. Use \\n for newlines inside string values, never literal newlines."},
                    {"role": "user", "content": prompt}
                ],
                response_format={"type": "json_object"},  # Forces valid JSON
                temperature=0.9,
                max_tokens=1500
            )
            text = response.choices[0].message.content.strip()
            parsed = json.loads(text)
            is_valid, reason = validate_example(parsed, cefr_level)

            if is_valid:
                return parsed
            else:
                print(f"  [REJECTED] {reason} (attempt {attempt+1})")

        except (json.JSONDecodeError, Exception) as e:
            print(f"  [ERROR] {e} (attempt {attempt+1})")
            time.sleep(0.5)

    return None


def get_vocab_for_level(vocab_entries, cefr_level):
    return [v for v in vocab_entries if v["cefr_level"] == cefr_level]


def format_vocab_sample(vocab_list, n=5):
    sample = random.sample(vocab_list, min(n, len(vocab_list)))
    lines = []
    for v in sample:
        gender_str = f" ({v['gender']})" if v['gender'] else ""
        lines.append(f"- {v['term']}{gender_str}: {v['definition']}")
    return '\n'.join(lines)


def format_vocab_context(vocab_list, n=5):
    sample = random.sample(vocab_list, min(n, len(vocab_list)))
    lines = []
    for v in sample:
        gender_str = f" ({v['gender']})" if v['gender'] else ""
        ex = v['examples'][0] if v['examples'] else ""
        lines.append(f"{v['term']}{gender_str} — {v['definition']} — Ex: {ex}")
    return '\n'.join(lines), sample

In [20]:
EXAMPLES_PER_RULE = 80  # ~80 x 25 rules = 2,000 grammar examples

grammar_examples = []
rejection_count = 0

for rule in tqdm(grammar_rules, desc="Grammar rules"):
    level = rule["cefr_level"]
    level_vocab = get_vocab_for_level(vocab_entries, level)
    if not level_vocab:
        level_vocab = get_vocab_for_level(vocab_entries, "A1")

    lang_rule = LANGUAGE_RULES.get(level, LANGUAGE_RULES["A2"])

    for i in range(EXAMPLES_PER_RULE):
        vocab_sample = format_vocab_sample(level_vocab)
        prompt = META_PROMPT_GRAMMAR.format(
            grammar_rule=rule["full_text"],
            vocab_sample=vocab_sample,
            cefr_level=level,
            language_rule=lang_rule
        )

        result = call_llm(prompt, level)
        if result:
            example = {
                "messages": [
                    {"role": "system",    "content": TUTOR_SYSTEM_PROMPT},
                    {"role": "user",      "content": result["user"]},
                    {"role": "assistant", "content": result["assistant"]}
                ]
            }
            grammar_examples.append(example)
        else:
            rejection_count += 1

        time.sleep(0.1)

    # Checkpoint
    with jsonlines.open(OUTPUT_FILE.replace(".jsonl", "_grammar_checkpoint.jsonl"), mode='w') as writer:
        writer.write_all(grammar_examples)

    print(f"  [{level}] {rule['title']}: {len(grammar_examples)} total, {rejection_count} rejected")

print(f"\nGenerated {len(grammar_examples)} grammar examples ({rejection_count} rejected)")


Grammar rules:   4%|▍         | 1/26 [04:59<2:04:40, 299.22s/it]

  [A1] Pronoms sujets et le verbe être: 80 total, 0 rejected


Grammar rules:   8%|▊         | 2/26 [09:05<1:47:10, 267.92s/it]

  [A1] Le verbe avoir et l'âge: 160 total, 0 rejected


Grammar rules:  12%|█▏        | 3/26 [13:30<1:42:17, 266.86s/it]

...

  [B2] La voix passive: 1920 total, 0 rejected
  [REJECTED] B2 response not French enough (67%) (attempt 1)


Grammar rules:  96%|█████████▌| 25/26 [2:10:37<05:13, 313.20s/it]

  [B2] Le discours indirect: 2000 total, 0 rejected
  [REJECTED] B2 response not French enough (67%) (attempt 1)
  [REJECTED] missing user or assistant key (attempt 1)
  [REJECTED] missing [CONTEXT] block in user (attempt 1)
  [REJECTED] missing user or assistant key (attempt 1)
  [REJECTED] missing user or assistant key (attempt 1)


Grammar rules: 100%|██████████| 26/26 [2:16:46<00:00, 315.64s/it]

  [B2] Pronoms relatifs : lequel, auquel, duquel: 2080 total, 0 rejected

Generated 2080 grammar examples (0 rejected)


In [23]:
from collections import defaultdict

VOCAB_EXAMPLES = 400
topics = defaultdict(list)
for v in vocab_entries:
    topics[v["topic"]].append(v)

vocab_examples = []
examples_per_topic = VOCAB_EXAMPLES // len(topics)

for topic, items in tqdm(topics.items(), desc="Vocab topics"):
    cefr = items[0]["cefr_level"]
    lang_rule = LANGUAGE_RULES.get(cefr, LANGUAGE_RULES["A2"])

    for i in range(examples_per_topic):
        context_text, sampled = format_vocab_context(items)
        prompt = META_PROMPT_VOCAB.format(
            vocab_items=context_text,
            cefr_level=cefr,
            language_rule=lang_rule
        )

        result = call_llm(prompt, cefr)
        if result:
            example = {
                "messages": [
                    {"role": "system",    "content": TUTOR_SYSTEM_PROMPT},
                    {"role": "user",      "content": result["user"]},
                    {"role": "assistant", "content": result["assistant"]}
                ]
            }
            vocab_examples.append(example)

        time.sleep(0.1)

print(f"Generated {len(vocab_examples)} vocabulary examples")

Vocab topics:   7%|▋         | 1/14 [02:09<28:06, 129.76s/it]

  [REJECTED] A1 response too French (55%) (attempt 1)
  [REJECTED] A1 response too French (55%) (attempt 1)
  [REJECTED] A1 response too French (64%) (attempt 1)


Vocab topics:  14%|█▍        | 2/14 [04:07<24:31, 122.64s/it]

  [ERROR] Expecting ':' delimiter: line 3 column 559 (char 914) (attempt 1)
  [ERROR] Expecting ':' delimiter: line 3 column 575 (char 889) (attempt 1)
  [REJECTED] A1 response too French (57%) (attempt 1)


Vocab topics:  21%|██▏       | 3/14 [06:44<25:22, 138.40s/it]

...

  [REJECTED] A1 response too French (55%) (attempt 1)
  [REJECTED] A1 response too French (56%) (attempt 1)
  [REJECTED] A1 response too French (70%) (attempt 1)


Vocab topics:  36%|███▌      | 5/14 [11:03<19:44, 131.63s/it]

  [REJECTED] A1 response too French (60%) (attempt 1)
  [REJECTED] A1 response too French (55%) (attempt 2)


Vocab topics: 100%|██████████| 14/14 [29:39<00:00, 127.07s/it]

Generated 392 vocabulary examples


In [24]:
import pandas as pd

all_examples = grammar_examples + vocab_examples
random.shuffle(all_examples)

with jsonlines.open(OUTPUT_FILE, mode='w') as writer:
    writer.write_all(all_examples)

print(f"Saved {len(all_examples)} examples to {OUTPUT_FILE}")

# Audit
has_reasoning_open = sum(1 for ex in all_examples if "<reasoning>" in ex["messages"][2]["content"])
has_reasoning_close = sum(1 for ex in all_examples if "</reasoning>" in ex["messages"][2]["content"])
has_content_after = sum(1 for ex in all_examples
    if "</reasoning>" in ex["messages"][2]["content"]
    and len(ex["messages"][2]["content"].split("</reasoning>")[-1].strip()) > 30)

print(f"\n=== AUDIT ===")
print(f"  <reasoning>:              {has_reasoning_open}/{len(all_examples)}")
print(f"  </reasoning>:             {has_reasoning_close}/{len(all_examples)}")
print(f"  Content after </reasoning>: {has_content_after}/{len(all_examples)}")

# Language distribution
level_lang = {"A1": [], "A2": [], "B1": [], "B2": [], "unknown": []}
french_markers = ['je ', 'tu ', 'il ', 'elle ', 'nous ', 'vous ', 'est ',
                  'sont ', 'une ', 'des ', 'dans ', 'pour ', 'avec ']
english_markers = ['the ', 'is ', 'are ', 'you ', 'your ', 'this ',
                   'try ', 'look ', 'think ', 'what ', 'how ', 'can ']

for ex in all_examples:
    user_msg = ex["messages"][1]["content"]
    assistant_msg = ex["messages"][2]["content"]
    visible = assistant_msg.split("</reasoning>")[-1].strip() if "</reasoning>" in assistant_msg else assistant_msg

    v_lower = visible.lower()
    fr = sum(v_lower.count(m) for m in french_markers)
    en = sum(v_lower.count(m) for m in english_markers)
    total = fr + en
    ratio = fr / total if total > 0 else 0.5

    level = "unknown"
    for lvl in ["A1", "A2", "B1", "B2"]:
        if f"({lvl})" in user_msg:
            level = lvl
            break
    level_lang[level].append(ratio)

print(f"\n=== LANGUAGE RATIO (visible response) ===")
print(f"  Target: A1=~20% French, A2=~50%, B1=~80%, B2=~100%")
for level in ["A1", "A2", "B1", "B2", "unknown"]:
    ratios = level_lang[level]
    if ratios:
        avg = sum(ratios) / len(ratios)
        print(f"  {level}: avg French = {avg:.0%}  (n={len(ratios)})")

# Preview
sample = random.choice(all_examples)
print(f"\n=== RANDOM EXAMPLE ===")
print(f"USER:\n{sample['messages'][1]['content'][:300]}")
print(f"\nASSISTANT:\n{sample['messages'][2]['content'][:500]}")

Saved 2472 examples to /content/drive/MyDrive/MrWorldwide/training_data_v3.jsonl

=== AUDIT ===
  <reasoning>:              2472/2472
  </reasoning>:             2472/2472
  Content after </reasoning>: 2472/2472

=== LANGUAGE RATIO (visible response) ===
  Target: A1=~20% French, A2=~50%, B1=~80%, B2=~100%
  A1: avg French = 12%  (n=1)
  B2: avg French = 89%  (n=14)
  unknown: avg French = 33%  (n=2457)

=== RANDOM EXAMPLE ===
USER:
[CONTEXT] Le discours indirect rapporte les paroles d'une personne sans les citer mot pour mot; il entraîne généralement des changements de temps, de pronoms et d'adverbes de temps/lieu (concordance des temps). Quand le verbe introducteur est au passé, on décale souvent les temps (présent → imparfai

ASSISTANT:
<reasoning>
The student used 'vient' in the present tense instead of the imparfait, which is required according to the concordance des temps when the introductory verb is in the past. The context clearly states that the present should change to impar

In [25]:
# Better CEFR detection — check the grammar content, not just "(A1)"
level_lang = {"A1": [], "A2": [], "B1": [], "B2": [], "unknown": []}

# Map grammar rule titles to their levels
level_keywords = {
    "A1": ["être", "avoir", "l'âge", "articles et genre", "-er", "négation", "ne ... pas", "pluriel", "poser des questions", "intonation"],
    "A2": ["-ir", "-re", "passé composé", "imparfait", "futur proche", "pronominaux", "adjectifs", "possessifs", "démonstratifs", "partitifs"],
    "B1": ["compléments", "pronoms y", "pronoms en", "relatifs qui que", "futur simple", "conditionnel", "subjonctif présent"],
    "B2": ["subjonctif passé", "voix passive", "discours indirect", "lequel", "auquel", "duquel"]
}

for ex in all_examples:
    ctx = ex["messages"][1]["content"].lower()
    assistant = ex["messages"][2]["content"]
    visible = assistant.split("</reasoning>")[-1].strip() if "</reasoning>" in assistant else assistant

    # Detect level from context content
    level = "unknown"
    for lvl, keywords in level_keywords.items():
        if any(kw.lower() in ctx for kw in keywords):
            level = lvl
            break

    v_lower = visible.lower()
    fr = sum(v_lower.count(m) for m in french_markers)
    en = sum(v_lower.count(m) for m in english_markers)
    total = fr + en
    ratio = fr / total if total > 0 else 0.5
    level_lang[level].append(ratio)

print("=== ACTUAL LANGUAGE RATIOS ===")
print("Target: A1=~20%, A2=~50%, B1=~80%, B2=~100%")
for level in ["A1", "A2", "B1", "B2", "unknown"]:
    ratios = level_lang[level]
    if ratios:
        avg = sum(ratios) / len(ratios)
        print(f"  {level}: avg French = {avg:.0%}  (n={len(ratios)})")

=== ACTUAL LANGUAGE RATIOS ===
Target: A1=~20%, A2=~50%, B1=~80%, B2=~100%
  A1: avg French = 24%  (n=1292)
  A2: avg French = 43%  (n=397)
  B1: avg French = 80%  (n=162)
  B2: avg French = 88%  (n=98)
  unknown: avg French = 25%  (n=523)
